# reguq Colab Workflow Check (Final v1)

This notebook validates the full `reguq` workflow on your `check` folder data and uses a one-time Colab bootstrap with auto-restart.

Expected files:
- `/content/check/train.csv` and `/content/check/test.csv`, or
- `/content/drive/MyDrive/check/train.csv` and `/content/drive/MyDrive/check/test.csv`


In [ ]:
# Install latest package from GitHub (contains bootstrap helper)
%pip -q install "git+https://github.com/DaneshSelwal/reguq.git"


In [ ]:
# One-time Colab bootstrap (pinned deps + auto restart)
from reguq import bootstrap_colab_environment

# This cell will restart runtime once on first run.
bootstrap_colab_environment(repo_url="https://github.com/DaneshSelwal/reguq.git")
print("Environment already prepared. Continue.")


In [ ]:
# Optional: mount Google Drive
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import pandas as pd

from reguq import (
    run_tuning,
    run_quantile,
    run_probabilistic,
    run_conformal_standard,
    run_from_config,
)

candidate_dirs = [
    Path('/content/check'),
    Path('/content/drive/MyDrive/check'),
]

DATA_DIR = None
for c in candidate_dirs:
    if (c / 'train.csv').exists() and (c / 'test.csv').exists():
        DATA_DIR = c
        break

if DATA_DIR is None:
    raise FileNotFoundError('Could not find train.csv/test.csv in /content/check or /content/drive/MyDrive/check')

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'

print('Using data directory:', DATA_DIR)


In [ ]:
# Set target column. If None, last column is used.
TARGET_COL = None  # Example: 'target'

train_preview = pd.read_csv(TRAIN_PATH)
if TARGET_COL is None:
    TARGET_COL = train_preview.columns[-1]

print('Detected target column:', TARGET_COL)
print('Train shape:', train_preview.shape)
display(train_preview.head())


## 1) Quick smoke test (fast)


In [ ]:
quick_result = run_quantile(
    data={'train_path': str(TRAIN_PATH), 'test_path': str(TEST_PATH)},
    target_col=TARGET_COL,
    models=['lightgbm'],
    params_source={'mode': 'defaults'},
    output_config={
        'output_dir': '/content/check_outputs/quick',
        'export_excel': True,
        'export_plots': True,
        'embed_excel_charts': True,
        'show_inline_plots': True,
        'chart_detail_level': 'detailed',
        'legend_position': 'upper right',
        'save_json': True,
    },
)

quick_result.metrics


## 2) Full workflow test (with detailed charts)


In [ ]:
config = {
    'data': {
        'train_path': str(TRAIN_PATH),
        'test_path': str(TEST_PATH),
        'target_col': TARGET_COL,
    },
    'models': ['lightgbm', 'xgboost', 'catboost'],
    'phases': ['quantile', 'probabilistic', 'conformal_standard'],
    'params_source': {'mode': 'defaults'},
    'conformal_standard': {
        'alpha': 0.1,
        'methods': ['mapie', 'puncc'],
        'mapie_method': 'plus',
    },
    'output': {
        'output_dir': '/content/check_outputs/full',
        'export_excel': True,
        'export_plots': True,
        'embed_excel_charts': True,
        'show_inline_plots': True,
        'chart_detail_level': 'detailed',
        'legend_position': 'upper right',
        'save_json': True,
    },
}

run_result = run_from_config(config)
print('Run ID:', run_result.run_id)
print('Output dir:', run_result.output_dir)
print('Completed phases:', list(run_result.results.keys()))


In [ ]:
# List generated artifacts
from pathlib import Path

out_dir = Path(run_result.output_dir)
for p in sorted(out_dir.rglob('*')):
    if p.is_file():
        print(p)


## 3) Optional tuning check (slower)


In [ ]:
# tuning_result = run_tuning(
#     data={'train_path': str(TRAIN_PATH), 'test_path': str(TEST_PATH)},
#     target_col=TARGET_COL,
#     models=['lightgbm'],
#     tuning_config={'n_trials': 5, 'cv': 3, 'random_state': 42},
#     output_config={
#         'output_dir': '/content/check_outputs/tuning',
#         'export_excel': True,
#         'export_plots': True,
#         'embed_excel_charts': True,
#         'show_inline_plots': True,
#         'chart_detail_level': 'detailed',
#         'legend_position': 'upper right',
#         'save_json': True,
#     },
# )
# tuning_result.summary
